# Reproduce the study in Google Colab

This notebook runs the complete pipeline on a Colab GPU. Choose what you need:

| Level | What it does | Time | Needs GPU |
|---|---|---|---|
| **A. Verify the paper's numbers** | Rebuilds every table and figure of the Results from the archived predictions in `results/predictions` | ~1 min | no |
| **B. Rerun the experiment** | 30 repeated splits × 8 strategies → new prediction archive, then Level A on it | ~7 h | yes (T4) |
| **C. Rerun the generator search** | The 100-trial Optuna searches that fixed the generator settings | many hours per strategy | yes |

Before you start: **Runtime → Change runtime type → T4 GPU** (only for B and C).

## 1. Get the code
Clones this repository into the Colab session.

In [ ]:
REPO_URL = 'https://github.com/alexlas73/PdM-with-WGAN-GP-sampling.git'
!git clone -q $REPO_URL repo
%cd repo
!git log -1 --format='commit %h  (%cd)'

## 2. Install the pinned libraries
After this cell finishes, use **Runtime → Restart session**, then continue with the next cell (no need to rerun this one).

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
%cd /content/repo
!python -m ftaa.versions

## Level A — verify the paper's numbers from the archived predictions

In [ ]:
!python -m ftaa.analysis --run results/predictions --out analysis_check
!python -m pytest -q tests/test_analysis.py

## Level B — rerun the experiment

**Data.** Download `predictive_maintenance.csv` from Kaggle (see `data/README.md`) and put it in your Google Drive (My Drive).
The output folder is on Drive, so an interrupted run can be resumed by running the same cell again: finished
(split, strategy) files are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/predictive_maintenance.csv'
OUT  = '/content/drive/MyDrive/ftaa_run'
!python -m ftaa.check_data $DATA

In [ ]:
# Quick test first (one split, fast strategies only, ~1 minute):
!python -m ftaa.experiment --data $DATA --out /content/quick_test --splits 1 --methods Baseline RUS SMOTE

In [ ]:
# Full run (~7 hours on a T4). Keep the browser tab open; if the session disconnects, reconnect,
# rerun cells 1-2 (and the Drive cell), then this cell again -- it resumes where it stopped.
!python -m ftaa.experiment --data $DATA --out $OUT

In [ ]:
!python -m ftaa.analysis --run $OUT --out $OUT/analysis

GPU training of the generators is not bit-for-bit deterministic, so a rerun reproduces the generative strategies'
results closely but not to the last digit; the non-generative strategies are deterministic. Compare
`$OUT/analysis/performance.csv` with `results/tables/performance.csv`.

### Figures 9 and 10 (t-SNE illustration of split 1, ~13 minutes)

In [ ]:
!python -m ftaa.tsne --data $DATA --settings results/predictions/settings.csv --split 1 --out $OUT/figures

## Level C — rerun the generator search (optional, long)
The `--storage` file on Drive lets an interrupted search resume.

In [ ]:
!python -m ftaa.hpo --data $DATA --strategy multi  --out $OUT/hpo --storage sqlite:///$OUT/hpo/multi.db
!python -m ftaa.hpo --data $DATA --strategy single --out $OUT/hpo --storage sqlite:///$OUT/hpo/single.db